# 10.3 Modifying data

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Many modeling projects involve solving a series of problem instances, each defined by somewhat different data. We describe here AMPL's facilities for resetting parameter values while leaving the model as is. They include commands for resetting data and for resampling random parameters, as well as the direct assignment of new values from Python.

We use the diet model of [Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1), `dietu.mod`, in which the nutrients with minimum and maximum requirements form two separate sets:

In [2]:
%%writefile dietu.mod

set MINREQ;                     # nutrients with minimum requirements
set MAXREQ;                     # nutrients with maximum requirements
set NUTR = MINREQ union MAXREQ; # nutrients
set FOOD;                       # foods

param cost {FOOD} > 0;
param f_min {FOOD} >= 0;
param f_max {j in FOOD} >= f_min[j];

param n_min {MINREQ} >= 0;
param n_max {MAXREQ} >= 0;

param amt {NUTR,FOOD} >= 0;

var Buy {j in FOOD} >= f_min[j], <= f_max[j];

minimize Total_Cost: sum {j in FOOD} cost[j] * Buy[j];

subject to Diet_Min {i in MINREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];

subject to Diet_Max {i in MAXREQ}:
   sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

Writing dietu.mod


In [3]:
import pandas as pd

food_df = pd.DataFrame(
    [
        ("BEEF", 3.19, 2, 10),
        ("CHK",  2.59, 2, 10),
        ("FISH", 2.29, 2, 10),
        ("HAM",  2.89, 2, 10),
        ("MCH",  1.89, 2, 10),
        ("MTL",  1.99, 2, 10),
        ("SPG",  1.99, 2, 10),
        ("TUR",  2.49, 2, 10),
    ],
    columns=["FOOD", "cost", "f_min", "f_max"],
).set_index("FOOD")

MINREQ = ["A", "B1", "B2", "C", "CAL"]
MAXREQ = ["A", "NA", "CAL"]

n_min = {"A": 700, "B1": 0, "B2": 0, "C": 700, "CAL": 16000}
n_max = {"A": 20000, "NA": 50000, "CAL": 24000}

# Nutrient content: one row per nutrient, one column per food
amt_df = pd.DataFrame(
    {
        "BEEF": [60, 20, 10, 15,  938, 295],
        "CHK":  [ 8,  0, 20, 20, 2180, 770],
        "FISH": [ 8, 10, 15, 10,  945, 440],
        "HAM":  [40, 40, 35, 10,  278, 430],
        "MCH":  [15, 35, 15, 15, 1182, 315],
        "MTL":  [70, 30, 15, 15,  896, 400],
        "SPG":  [25, 50, 25, 15, 1329, 370],
        "TUR":  [60, 20, 15, 10, 1397, 450],
    },
    index=["A", "C", "B1", "B2", "NA", "CAL"],
)
amt = amt_df.stack().to_dict()   # {(nutrient, food): amount}

In [4]:
ampl.read("dietu.mod")

ampl.set_data(food_df, "FOOD")
ampl.set["MINREQ"] = MINREQ
ampl.set["MAXREQ"] = MAXREQ
ampl.param["n_min"] = n_min
ampl.param["n_max"] = n_max
ampl.param["amt"] = amt

ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1: optimal solution; objective 74.27382022
2 simplex iterations
0 barrier iterations


## 10.3.1 Resetting

To delete the current data for several model components, without changing the current model itself, use the AMPL `reset data` command, as in:

In [5]:
ampl.eval("reset data MINREQ, MAXREQ, amt, n_min, n_max;")

You may then assign new values for these sets and parameters from Python. To delete all data, use `reset data` without a list of components.

The `update data` command works similarly, but does not actually delete any data until new values are assigned. Thus if you use `update data` for the same components, but only assign new values for `MINREQ`, `MAXREQ`, `amt` and `n_min`, the previous values for `n_max` will remain. Because we used `reset data` instead, `n_max` is now without values, and we get an error message when we next try to solve:

In [6]:
ampl.set["MINREQ"] = MINREQ
ampl.set["MAXREQ"] = MAXREQ
ampl.param["amt"] = amt
ampl.param["n_min"] = n_min

ampl.solve(solver="highs")
ampl.solve_result, ampl.solve_result_num

[ERROR] 
	Error executing "solve" command:
	error processing constraint Diet_Max['A']:
		no value for n_max['A']


('?', -1)

As explained in [Section 10.2](10_2_setting_up_and_solving_models_and_data.ipynb), the error does not stop the Python program; the `'?'` result tells us that no problem could be generated. Once the values of `n_max` are supplied, the problem can be solved again:

In [7]:
ampl.param["n_max"] = n_max

ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1: optimal solution; objective 74.27382022
0 simplex iterations
0 barrier iterations


Assignments from Python to a parameter work like `update data`: values are replaced only for the members that are assigned, and the values for all other members remain. Here we raise the requirement for vitamin C only:

In [8]:
ampl.param["n_min"] = {"C": 800}
ampl.display("n_min")

n_min [*] :=
  A    700
 B1      0
 B2      0
  C    800
CAL  16000
;



In [9]:
ampl.param["n_min"] = {"C": 700}   # back to the original requirement

## 10.3.2 Resampling

The `reset data` command also acts to resample the randomly computed parameters described in [Section 7.6](../07/7_6_randomly_generated_parameters.ipynb). Continuing with the variant of `steel4.mod` introduced in that section, if the definition of parameter `avail` is changed so that its value is given by a random function:

In [10]:
%%writefile steel4r.mod

set PROD;   # products
set STAGE;  # stages

param rate {PROD,STAGE} > 0; # tons per hour in each stage
param profit {PROD};         # profit per ton

param commit {PROD} >= 0;    # lower limit on tons sold in week
param market {PROD} >= 0;    # upper limit on tons sold in week

param avail_mean {STAGE} >= 0;
param avail_variance {STAGE} >= 0;
param avail {s in STAGE} =
   Normal(avail_mean[s], avail_variance[s]);

var Make {p in PROD} >= commit[p], <= market[p]; # tons produced

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];

subject to Time {s in STAGE}:
   sum {p in PROD} (1/rate[p,s]) * Make[p] <= avail[s];

Writing steel4r.mod


with corresponding data:

In [11]:
steel = AMPL()   # a second AMPL session, for the steel model
steel.read("steel4r.mod")

steel.set_data(
    pd.DataFrame(
        [("bands", 25, 1000, 6000), ("coils", 30, 500, 4000), ("plate", 29, 750, 3500)],
        columns=["PROD", "profit", "commit", "market"],
    ).set_index("PROD"),
    "PROD",
)
steel.set["STAGE"] = ["reheat", "roll"]
steel.param["rate"] = {
    ("bands", "reheat"): 200, ("bands", "roll"): 200,
    ("coils", "reheat"): 200, ("coils", "roll"): 140,
    ("plate", "reheat"): 200, ("plate", "roll"): 160,
}
steel.param["avail_mean"] = {"reheat": 35, "roll": 40}
steel.param["avail_variance"] = {"reheat": 5, "roll": 2}

then AMPL will take new samples from the Normal distribution after each `reset data`. Different samples result in different solutions, and hence in different optimal objective values:

In [12]:
steel.solve(solver="highs")
assert steel.solve_result == "solved", steel.solve_result
steel.display("avail")

HiGHS 1.15.1: optimal solution; objective 187632.2489
2 simplex iterations
0 barrier iterations
avail [*] :=
reheat  32.3504
  roll  43.038
;



In [13]:
steel.eval("reset data avail;")

steel.solve(solver="highs")
assert steel.solve_result == "solved", steel.solve_result
steel.display("avail")

HiGHS 1.15.1: optimal solution; objective 158882.901
2 simplex iterations
0 barrier iterations
avail [*] :=
reheat  32.0306
  roll  32.6855
;



Only `reset data` has this effect; if you call `reset()` then AMPL's random number generator is reset, and the values of `avail` repeat from the beginning. ([Section 7.6](../07/7_6_randomly_generated_parameters.ipynb) explains how to reset the generator's "seed" to get a different sequence of random numbers.)

In [14]:
steel.close()

## 10.3.3 Assigning new values

AMPL's `let` command permits you to change particular data values while leaving the model the same. From Python, the same is done by assigning to `ampl.param[...]` or `ampl.set[...]`, which is more convenient for small or easy-to-describe changes than `reset data` or `update data`. You can use it, for example, to solve the diet model again, trying out a series of upper bounds `f_max["CHK"]` on the purchases of food `CHK`:

In [15]:
ampl.param["f_max"]["CHK"] = 11
ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1: optimal solution; objective 73.43818182
1 simplex iterations
0 barrier iterations


In [16]:
ampl.param["f_max"]["CHK"] = 12
ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

HiGHS 1.15.1: optimal solution; objective 73.43818182
0 simplex iterations
0 barrier iterations


Relaxing the bound to 11 reduces the cost somewhat, but further relaxation apparently has no benefit.

A whole collection of values can be changed at once by assigning a dictionary, with one entry for each member that is to be changed. You could use this feature to change all upper bounds to 8:

In [17]:
ampl.param["f_max"] = {j: 8 for j in food_df.index}

or, reading the current values with `to_dict()`, to increase all upper bounds by 10 percent:

In [18]:
f_max = ampl.param["f_max"].to_dict()
ampl.param["f_max"] = {j: 1.1 * f_max[j] for j in f_max}
ampl.display("f_max")

f_max [*] :=
BEEF  8.8
 CHK  8.8
FISH  8.8
 HAM  8.8
 MCH  8.8
 MTL  8.8
 SPG  8.8
 TUR  8.8
;



In general, any set or parameter whose declaration does not define it using an `=` phrase may be assigned in this way, while the value assigned may be any appropriate Python value. A set or parameter that is defined by an `=` phrase, such as `NUTR` in `dietu.mod`, cannot be assigned; amplpy raises an error if you try.

Although AMPL does not impose any other restrictions on what you can change, you should take care in changing any set or parameter that affects the indexing of other data.

For example, after solving the multiperiod production problem `steelT` of [Section 4.2](../04/4_2_a_multiperiod_production_model.ipynb), it might be tempting to change the number of weeks `T` from 4 (as given in the original data) to 3:

In [19]:
%%writefile steelT.mod

set PROD;     # products
param T > 0;  # number of weeks

param rate {PROD} > 0;          # tons per hour produced
param inv0 {PROD} >= 0;         # initial inventory
param avail {1..T} >= 0;        # hours available in week
param market {PROD,1..T} >= 0;  # limit on tons sold in week

param prodcost {PROD} >= 0;     # cost per ton produced
param invcost {PROD} >= 0;      # carrying cost/ton of inventory
param revenue {PROD,1..T} >= 0; # revenue per ton sold

var Make {PROD,1..T} >= 0;      # tons produced
var Inv {PROD,0..T} >= 0;       # tons inventoried
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t]; # tons sold

# Total revenue less costs in all weeks
maximize Total_Profit:
   sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
      prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t]);

# Total of hours used by all products
# may not exceed hours available, in each week
subject to Time {t in 1..T}:
   sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

# Initial inventory must equal given value
subject to Init_Inv {p in PROD}:  Inv[p,0] = inv0[p];

# Tons produced and taken from inventory
# must equal tons sold and put into inventory
subject to Balance {p in PROD, t in 1..T}:
   Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

Writing steelT.mod


In [20]:
import pandas as pd

steelT_prod = pd.DataFrame(
    [
        ["bands", 200, 10, 10, 2.5],
        ["coils", 140,  0, 11, 3],
    ],
    columns=["PROD", "rate", "inv0", "prodcost", "invcost"],
).set_index("PROD")

steelT_week = pd.DataFrame(
    [
        ["bands", 1, 25, 6000],
        ["bands", 2, 26, 6000],
        ["bands", 3, 27, 4000],
        ["bands", 4, 27, 6500],
        ["coils", 1, 30, 4000],
        ["coils", 2, 35, 2500],
        ["coils", 3, 37, 3500],
        ["coils", 4, 39, 4200],
    ],
    columns=["PROD", "T", "revenue", "market"],
).set_index(["PROD", "T"])

steelT_avail = {1: 40, 2: 40, 3: 32, 4: 40}

In [21]:
steel = AMPL()   # a new session for the steelT model
steel.read("steelT.mod")
steel.param["T"] = 4
steel.param["avail"] = steelT_avail
steel.set_data(steelT_prod, "PROD")
steel.set_data(steelT_week)

steel.solve(solver="highs")
assert steel.solve_result == "solved", steel.solve_result

HiGHS 1.15.1: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations


In [22]:
steel.param["T"] = 3
steel.solve(solver="highs")

[ERROR] 
	Error executing "solve" command:
	error processing param avail:
		invalid subscript avail[4] discarded.
[ERROR] 
	Error executing "solve" command:
	error processing param market:
		2 invalid subscripts discarded:
		market['bands',4]
		market['coils',4]
[ERROR] 
	Error executing "solve" command:
	error processing param revenue:
		2 invalid subscripts discarded:
		revenue['bands',4]
		revenue['coils',4]


HiGHS 1.15.1: optimal solution; objective 363433
0 simplex iterations
0 barrier iterations


The problem here is that AMPL still has current data for 4th-week parameters such as `avail[4]`, which has become invalid with the change of `T` to 3. AMPL discards these values, with the error messages shown, and solves the 3-week problem that remains. The 4th-week data is then lost, however, so that changing `T` back to 4 leaves a problem that cannot be generated:

In [23]:
steel.param["T"] = 4
steel.solve(solver="highs")
steel.solve_result, steel.solve_result_num

[ERROR] 
	Error executing "solve" command:
	error processing var Sell['bands',4]:
		no value for market['bands',4]


('?', -1)

In [24]:
steel.close()

If you want to properly reduce the number of weeks in the linear program while using the same data, you must declare two parameters:

```ampl
param Tdata integer > 0;
param T integer <= Tdata;
```

Use `1..Tdata` for indexing in the `param` declarations, while retaining `1..T` for the variables, objective and constraints:

In [25]:
%%writefile steelTd.mod

set PROD;              # products
param Tdata integer > 0;    # number of weeks of data
param T integer <= Tdata;   # number of weeks in the model

param rate {PROD} > 0;              # tons per hour produced
param inv0 {PROD} >= 0;             # initial inventory
param avail {1..Tdata} >= 0;        # hours available in week
param market {PROD,1..Tdata} >= 0;  # limit on tons sold in week

param prodcost {PROD} >= 0;         # cost per ton produced
param invcost {PROD} >= 0;          # carrying cost/ton of inventory
param revenue {PROD,1..Tdata} >= 0; # revenue per ton sold

var Make {PROD,1..T} >= 0;      # tons produced
var Inv {PROD,0..T} >= 0;       # tons inventoried
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t]; # tons sold

maximize Total_Profit:
   sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
      prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t]);

subject to Time {t in 1..T}:
   sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

subject to Init_Inv {p in PROD}:  Inv[p,0] = inv0[p];

subject to Balance {p in PROD, t in 1..T}:
   Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

Writing steelTd.mod


Then you can assign `T` as you like:

In [26]:
steel = AMPL()
steel.read("steelTd.mod")
steel.param["Tdata"] = 4
steel.param["avail"] = steelT_avail
steel.set_data(steelT_prod, "PROD")
steel.set_data(steelT_week)

for weeks in [4, 3, 4]:
    steel.param["T"] = weeks
    steel.solve(solver="highs")
    assert steel.solve_result == "solved", steel.solve_result

steel.close()

HiGHS 1.15.1: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 363433
0 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 515033
3 simplex iterations
0 barrier iterations


You can also assign new values to variables from Python. This is sometimes a convenient feature for exploring an alternative solution. For example, here is what happens when we solve for the optimal diet as above, then round the optimal solution down to the next lowest integer number of packages:

In [27]:
import math

ampl.param["f_max"] = food_df["f_max"].to_dict()   # restore the original bounds
ampl.solve(solver="highs")
assert ampl.solve_result == "solved", ampl.solve_result

buy = ampl.var["Buy"].to_dict()
ampl.var["Buy"] = {j: math.floor(buy[j]) for j in buy}
ampl.display("Total_Cost")
ampl.display("n_min", "Diet_Min.slack")

HiGHS 1.15.1: optimal solution; objective 74.27382022
1 simplex iterations
0 barrier iterations
Total_Cost = 73.29

:     n_min Diet_Min.slack    :=
A       700        291
B1        0        595
B2        0        485
C       700        -20
CAL   16000       -190
;



Because we have assigned new values to the variables, the objective and the slacks are automatically computed from the new, rounded values. The cost has dropped by about \$1, but the solution is now short of the requirements for C by nearly 3 percent and for CAL by about 1 percent.

<a id='table-10-1'></a>

| AMPL function | Python equivalent | result |
| :--- | :--- | :--- |
| `ceil(x)` | `math.ceil(x)` | ceiling of x (next higher integer) |
| `floor(x)` | `math.floor(x)` | floor of x (next lower integer) |
| `precision(x, n)` | `float(f"{x:.{n}g}")` | x rounded to n significant digits |
| `round(x, n)` | `round(x, n)` | x rounded to n digits past the decimal point |
| `round(x)` | `round(x)` | x rounded to the nearest integer |
| `trunc(x, n)` | `math.trunc(x * 10**n) / 10**n` | x truncated to n digits past the decimal point |
| `trunc(x)` | `math.trunc(x)` | x truncated to an integer (fractional part dropped) |

**[Table 10-1](#table-10-1):** Rounding functions.

AMPL provides a variety of rounding functions that can be used in this way; they are summarized in [Table 10-1](#table-10-1), together with Python expressions that compute the same values. The AMPL functions can be used in any AMPL expression, including those evaluated from Python:

In [28]:
ampl.get_value("round(Total_Cost, 1)")

73.3

The Python versions are convenient when the values have already been retrieved, as in the rounding example above. (Python's `round()` rounds a value exactly halfway between two integers to the even one, so `round(2.5)` is 2.)